### Importeer de dataset

In [1]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Trainset: {X_train.shape}  |  Testset: {X_test.shape}")
print(f"Label min: {y_train.min()}, max: {y_train.max()}")


Trainset: (60000, 28, 28)  |  Testset: (10000, 28, 28)
Label min: 0, max: 9


We gaan nu de sklearn library voor het neurale netwerk gebruiken om te runnen op onze dataset. 

Je kunt in de huiswerkopdracht van deze week kijken voor hints ( opdracht 3.2 )

### Labels representatie

Bedenk het volgende voor je mnist dataset:


- In welke range zijn de labels nu (wat is de min/max waarde)?

  De labels liggen in de range **0 t/m 9** (min=0, max=9), één waarde per cijfer.

- Zijn de waardes numeriek of eigenlijk nominaal? Waarom?

  De waardes zijn **nominaal**. Cijfer 3 is niet "groter" of "kleiner" dan cijfer 2 — het zijn categorieën zonder onderlinge ordening of rekenkundige betekenis.

- Waarom willen we dit omzetten naar een ander formaat?

  Een neuraal netwerk zou anders aannemen dat label 9 negen keer zo "groot" is als label 1, wat fout is. Door OneHotEncoding te gebruiken behandelen we elke klasse als een aparte binaire kolom, zonder valse ordening.

We willen de **labels** omzetten naar een formaat die een NN beter kan interpreteren. Gebruik je hiervoor de OneHot Encoder?

Ja, we gebruiken de **OneHotEncoder**. Die zet elk label om naar een vector van 10 posities waarbij alleen de positie van het juiste cijfer een 1 heeft en de rest 0. Zo leert de output-laag voor elke klasse een aparte kans te voorspellen.

In [2]:
# Labels 0-9 zijn nominaal: cijfer 3 is niet "groter" dan cijfer 2
# OneHotEncoder zet bijv. label 3 om naar [0,0,0,1,0,0,0,0,0,0]
encoder = OneHotEncoder(sparse_output=False)

y_train_ohe = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_ohe  = encoder.transform(y_test.reshape(-1, 1))

print(f"Origineel label : {y_train[0]}")
print(f"One-hot encoded : {y_train_ohe[0]}")
print(f"Shape y_train_ohe: {y_train_ohe.shape}")


Origineel label : 5
One-hot encoded : [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
Shape y_train_ohe: (60000, 10)


### Split test /training set

Ook hier moet je de test en de training set opspliten. Denk ook na over de representatie van je data. 


Als je pixelwaardes als parameters gebruikt, hoe ga je ze encoden?

We passen twee stappen toe:
1. **Flatten**: elk beeld van 28×28 pixels wordt omgezet naar een platte vector van **784 waarden**, zodat het als invoer voor een Dense-laag kan dienen.
2. **Normaliseren**: de pixelwaarden (0–255) worden gedeeld door 255, zodat alle waarden in de range **0–1** vallen. Dit zorgt voor stabielere en snellere training.

In [3]:
# Pixels platmaken: 28x28 → 784, en normaliseren naar 0-1
X_train_flat = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test_flat  = X_test.reshape(-1, 784).astype('float32') / 255.0

print(f"X_train shape : {X_train_flat.shape}")
print(f"X_test  shape : {X_test_flat.shape}")
print(f"Pixel range   : {X_train_flat.min():.1f} - {X_train_flat.max():.1f}")


X_train shape : (60000, 784)
X_test  shape : (10000, 784)
Pixel range   : 0.0 - 1.0


### Simple NN: Layers opzet, architectuur

Gebruik keras van tensorflow om een neuraal netwerk op te zetten. De nieuwe manier om **keras** te gebruiken is:

    import tensorflow as tf
    model = tf.keras.Sequential([...])

Let op: voor sommigen zal de oude manier nog werken:


    from tensorflow import keras
    model = keras.Sequential([...])

Voeg de lagen van je neurale netwerk hier toe, dus tussen de blokhaken.
- Voeg de input layer toe. 

        tf.keras.Input(shape=(?,)),

Als parameter (?) kies het aantal input nodes. Hoeveel nodes kies je en waarom?

  **784 nodes**, want elk beeld is 28×28 pixels = 784 pixels. Elke pixel is een aparte input voor het netwerk.

- Voeg nu een hidden layer toe. Je kunt volgende code gebruiken

        tf.keras.layers.Dense("?", activation=?)

- Kies als eerste parameter ("?") het aantal nodes in deze layer. Hoeveel nodes kies je? Waarom?

  **128 nodes** in de eerste hidden layer, **64 nodes** in de tweede. Dit is een gangbare keuze die genoeg capaciteit geeft om patronen te leren zonder te zwaar te zijn.

- Kies de activation. Wat was activation ook al weer? Voor nu kun je gewoon de ***relu*** gebruiken.

  Een **activatiefunctie** bepaalt of en hoe sterk een neuron "vuurt" op basis van zijn invoer. **ReLU** (Rectified Linear Unit) geeft max(0, x) terug: negatieve waarden worden 0, positieve blijven onveranderd. Dit voorkomt het verdwijnen van gradiënten.

- Je kunt nog meer hidden layers toevoegen. Hoeveel hidden layers voeg je toe?

  **2 hidden layers** — dit geeft het netwerk genoeg diepte om abstracte patronen (zoals cijfervormen) te leren.

- Voeg nu de output layer toe. Code voor de output layer is hetzelfde als hidden, maar met een andere activatie functie. Hoeveel nodes kies je? Kies voor activation="softmax". Wat doet softmax ook al weer?

  **10 nodes** (één per cijfer 0–9). **Softmax** zet de ruwe uitvoer om naar kansen die optellen tot 1. De klasse met de hoogste kans is de voorspelling.

In [4]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax'),
])


### Compileer je model

Gebruik onderstaande code om je model te **compileren**

Wat compile doet: 

- Bepaalt hoe het model leert: adam optimizer
- Bepaalt wat het model probeert te minimaliseren: categorical crossentropy
- Bepaalt wat je wilt zien als prestatie‑indicator: 'accuracy' (bij ?)

Let er op dat je "loss" er niet bij hoeft te zetten omdat die zowieso als metric gebruikt wordt. 

- Wat is accuracy? Wat kun je er verder bij zetten?

  **Accuracy** is het percentage voorbeelden dat het model correct classificeert. Je kunt er ook **precision** en **recall** bij zetten: precision meet hoe vaak een voorspelde klasse echt klopt, recall meet hoeveel van de echte klasse-voorbeelden gevonden worden.

De dingen die je hier instelt worden pas later gebruikt, namelijk in de volgende stap - als je 

    .fit 

doet.

In [5]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


### Model trainen

Nu gaan we het model trainen. Bepaal hoeveel epochs je wilt doen en eg hoe groot de batch grootte is.

In [6]:
history = model.fit(X_train_flat, y_train_ohe, epochs=10, batch_size=32, verbose=1)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9307 - loss: 0.2393
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 966us/step - accuracy: 0.9680 - loss: 0.1026
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 881us/step - accuracy: 0.9782 - loss: 0.0706
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 883us/step - accuracy: 0.9830 - loss: 0.0540
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 916us/step - accuracy: 0.9864 - loss: 0.0430
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 927us/step - accuracy: 0.9884 - loss: 0.0359
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9911 - loss: 0.0279
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 981us/step - accuracy: 0.9921 - loss: 0.0243
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 902us/step - accuracy: 0.9921 - loss: 0.0229
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 873us/step - accuracy: 0.9935 - loss: 0.0194


### Maak voorspellingen

Nu kunnen we voorspellingen maken. Gebruik de functie 

    .predict()



In [9]:
pred = model.predict(X_test_flat[:1])

print("Kansen per klasse:")
for i, p in enumerate(pred[0]):
    print(f"Klasse {i}: {p:.6f}")

voorspeld = np.argmax(pred[0])
echt = np.argmax(y_test_ohe[0])

print(f"\nVoorspelde klasse: {voorspeld} vs echte klasse: {echt}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Kansen per klasse:
Klasse 0: 0.000000
Klasse 1: 0.000000
Klasse 2: 0.000000
Klasse 3: 0.000007
Klasse 4: 0.000000
Klasse 5: 0.000000
Klasse 6: 0.000000
Klasse 7: 0.999993
Klasse 8: 0.000000
Klasse 9: 0.000000

Voorspelde klasse: 7 vs echte klasse: 7


In [ ]:
loss, acc = model.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f"Test loss    : {loss:.4f}")
print(f"Test accuracy: {acc:.4f}")

### Experimenteer en onderzoek

Experimenteer nu zelf met verschillende settings / aantal hidden nodes / layers / activation functions en andere. Kijk of je algoritme beter en sneller kan voorspellen.

In [8]:
model2 = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax'),
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model2.fit(X_train_flat, y_train_ohe, epochs=10, batch_size=64, verbose=1)


Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9318 - loss: 0.2311
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9716 - loss: 0.0947
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9804 - loss: 0.0624
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9856 - loss: 0.0456
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9890 - loss: 0.0347
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9916 - loss: 0.0268
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9926 - loss: 0.0225
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9930 - loss: 0.0210
Epoch 9/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9938 - loss: 0.0179
Epoch 10/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9959 - loss: 0.0124


In [ ]:
loss2, acc2 = model2.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f"Model 1 - Test accuracy: {acc:.4f}")
print(f"Model 2 - Test accuracy: {acc2:.4f}")
print(f"\nModel 2 is {'beter' if acc2 > acc else 'slechter of gelijk'} dan model 1.")